# SMART MEDICINAL PLANT IDENTIFICATION SYSTEM (SMPIS)

# libraries

In [ ]:
# necessary dependencies
import numpy as np
import pandas as pd
import os
import random

# data processing
from PIL import Image, ImageOps
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.applications.vgg19 import preprocess_input

# model handling
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

# performance measurement
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# getting the data

In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eyadalhusseini/medicinal-plant-identification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'medicinal-plant-identification-dataset' dataset.
Path to dataset files: /kaggle/input/medicinal-plant-identification-dataset


In [11]:
os.listdir(path)

['Medicinal Plant Identification Dataset']

In [12]:
path = os.path.join(path, 'Medicinal Plant Identification Dataset')
print(os.listdir(path))

['Augmented-Images', 'Original-Images-Version-02']


In [13]:
path = os.path.join(path, 'Augmented-Images')
print(os.listdir(path))

['Marsh Pennywort Leaf', 'Mint Leaf', 'Neem Leaf', 'Curry Leaf', 'Arjun Leaf', 'Rubble Leaf']


In [ ]:
imgexts = {".jpg", ".jpeg", ".png"}

In [41]:
total = 0
counts = {}

for folder in os.listdir(path):
    folderpath = os.path.join(path, folder)

    if not os.path.isdir(folderpath):
        continue

    images = [
        f for f in os.listdir(folderpath)
        if os.path.splitext(f)[1].lower() in imgexts
    ]

    count = len(images)
    counts[folder] = count
    total += count

    print(f"{folder}: {count} images")

print("\nTotal images:", total)

Marsh Pennywort Leaf: 1470 images
Mint Leaf: 1540 images
Neem Leaf: 490 images
Curry Leaf: 1155 images
Arjun Leaf: 1540 images
Rubble Leaf: 1925 images

Total images: 8120


In [15]:
"""
getting the data from the folders...
...as paths and label
"""
data = []
max = 490

for folder in os.listdir(path):
    # getting the folder path form path
    folderpath = os.path.join(path, folder)
    if not os.path.isdir(folderpath):
        continue
    # getting images and checking lower ? image exits
    images = [
        f for f in os.listdir(folderpath)
        if os.path.splitext(f)[1].lower() in imgexts
    ]

    # ensuring class balance
    # randomize order
    random.shuffle(images)
    # cap at max per folder
    images = images[:max]

    print(folder, "has", len(images), "images (capped)")

    for imgname in images:
      # getting the image path + folder path
        imgpath = os.path.join(folderpath, imgname)
        data.append((imgpath, folder))

print("Total loaded:", len(data))

Marsh Pennywort Leaf has 490 images (capped)
Mint Leaf has 490 images (capped)
Neem Leaf has 490 images (capped)
Curry Leaf has 490 images (capped)
Arjun Leaf has 490 images (capped)
Rubble Leaf has 490 images (capped)
Total loaded: 2940


### truning it into a dataframe

In [16]:
df = pd.DataFrame(data,columns=["paths", "label"])

In [17]:
df.sample(5)

,paths,label
184,/kaggle/input/medicinal-plant-identification-d...,Marsh Pennywort Leaf
1098,/kaggle/input/medicinal-plant-identification-d...,Neem Leaf
2099,/kaggle/input/medicinal-plant-identification-d...,Arjun Leaf
1313,/kaggle/input/medicinal-plant-identification-d...,Neem Leaf
141,/kaggle/input/medicinal-plant-identification-d...,Marsh Pennywort Leaf


In [18]:
"""
df[x]['label'] =  Mint Leaf --> df[x]['label'] =  Mint Leaf  = 4
"""
le = LabelEncoder()
df['labelencoded'] = le.fit_transform(df['label'])

In [19]:
df.sample(5)

,paths,label,labelencoded
1688,/kaggle/input/medicinal-plant-identification-d...,Curry Leaf,1
878,/kaggle/input/medicinal-plant-identification-d...,Mint Leaf,3
120,/kaggle/input/medicinal-plant-identification-d...,Marsh Pennywort Leaf,2
2111,/kaggle/input/medicinal-plant-identification-d...,Arjun Leaf,0
1328,/kaggle/input/medicinal-plant-identification-d...,Neem Leaf,4


# pre-processing the data

note: this cell takes a lot of time

In [20]:
"""
We go through the entire dataframe:
1- Making sure the images are RGB.
2- We use ImageOps to make sure all images are the same size.
3- We convert the images into a NumPy array.
4- Apply VGG19 preprocessing to x.
"""
target_size = (224, 224)

x = []
y = []

for row in df.itertuples(index=False):
    img = Image.open(row.paths).convert("RGB")
    img = ImageOps.pad(img, target_size)

    x.append(np.asarray(img, dtype=np.float32))
    y.append(row.labelencoded)

x = np.asarray(x, dtype=np.float32)
y = np.asarray(y)

x = preprocess_input(x)

In [21]:
print(x.shape,y.shape)
print(x.dtype)

(2940, 224, 224, 3) (2940,)
float32


### splting the data

In [22]:
xtrain, xtest,ytrain,ytest = train_test_split(
    x,y,
    test_size=0.2,
    stratify=y, # ensuring class balance
    random_state=42
)

In [23]:
print(xtrain.shape,ytrain.shape)
print(xtest.shape,ytest.shape)

(2352, 224, 224, 3) (2352,)
(588, 224, 224, 3) (588,)


# model time

### initializing the backbone

In [24]:
basemodel = VGG19(
    # weights = imagenet
    include_top = False,
    input_shape = (224,224,3)
)

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [25]:
"""
freezing the layers of the model(VGG19) itself
"""
for layer in basemodel.layers:
  layer.trainable = False

### making the classification head

In [26]:
x = basemodel.output # the yield of the output layer
x = Flatten()(x) # flatten so we can train a layer on our data
x = Dense(256,activation='relu')(x) # said layer
output = Dense(6,activation='softmax')(x) # the class classifier

### compiling the final model

In [27]:
# attaching the classification head
model = Model(inputs = basemodel.input,outputs=output)

In [28]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

# the Magnum opus

In [42]:
history = model.fit(
    xtrain,ytrain,
    validation_data = (xtest,ytest),
    batch_size=32,
    epochs=30
)

Epoch 1/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 273ms/step - accuracy: 1.0000 - loss: 1.6827e-08 - val_accuracy: 0.9898 - val_loss: 0.1371
Epoch 2/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 21s 284ms/step - accuracy: 1.0000 - loss: 1.6320e-08 - val_accuracy: 0.9898 - val_loss: 0.1370
Epoch 3/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 22s 295ms/step - accuracy: 1.0000 - loss: 1.5864e-08 - val_accuracy: 0.9898 - val_loss: 0.1369
Epoch 4/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 274ms/step - accuracy: 1.0000 - loss: 1.5357e-08 - val_accuracy: 0.9898 - val_loss: 0.1369
Epoch 5/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 271ms/step - accuracy: 1.0000 - loss: 1.4901e-08 - val_accuracy: 0.9898 - val_loss: 0.1368
Epoch 6/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 21s 286ms/step - accuracy: 1.0000 - loss: 1.4495e-08 - val_accuracy: 0.9898 - val_loss: 0.1368
Epoch 7/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 21s 284ms/step - accuracy: 1.0000 - loss: 1.4039e-08 - val_accuracy: 0.9898 - val_loss: 0.1367
Epoch 8/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 276ms/step - accuracy: 1.00

# performance measurement

In [43]:
ypred_probs = model.predict(xtest)

19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 215ms/step


In [44]:
ypred = np.argmax(ypred_probs, axis=1)

In [45]:
acc = accuracy_score(ytest, ypred)
print("Validation accuracy:", acc)

Validation accuracy: 0.9897959183673469


In [46]:
print(classification_report(
    ytest,
    ypred,
    target_names=le.classes_
))

                      precision    recall  f1-score   support

          Arjun Leaf       1.00      1.00      1.00        98
          Curry Leaf       0.95      1.00      0.98        98
Marsh Pennywort Leaf       1.00      1.00      1.00        98
           Mint Leaf       1.00      0.95      0.97        98
           Neem Leaf       0.99      0.99      0.99        98
         Rubble Leaf       1.00      1.00      1.00        98

            accuracy                           0.99       588
           macro avg       0.99      0.99      0.99       588
        weighted avg       0.99      0.99      0.99       588



# deployment

In [47]:
model.save('leaf_model.keras')